# Hemodynamic Response Functions

This notebook demonstrates how to work with Hemodynamic Response Functions (HRFs) in pyfmridesign.

Author: Bradley R. Buchsbaum (Python port)  
Date: 2024

## Introduction to Hemodynamic Response Functions (HRFs)

A hemodynamic response function (HRF) models the temporal evolution of the fMRI BOLD (Blood-Oxygen-Level-Dependent) signal in response to a brief neural event. Typically, the BOLD signal peaks 4-6 seconds after the event onset and then returns to baseline, often with a slight undershoot.

`pyfmridesign` provides tools to define, manipulate, and visualize various HRFs commonly used in fMRI analysis.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pyfmridesign import (
    HRF, gen_hrf, HRF_SPMG1, evaluate,
    regressor, samples, SamplingFrame
)

# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')

## Pre-defined HRF Objects

`pyfmridesign` includes several pre-defined HRF objects from the `pyfmrihrf` package. These are functions with specific attributes defining their type, number of basis functions (`nbasis`), and effective duration (`span`).

Let's look at the SPM canonical HRF (`HRF_SPMG1`), which is based on the difference of two gamma functions:

In [ ]:
# SPM canonical HRF
print(f"HRF type: {HRF_SPMG1.type}")
print(f"Number of basis functions: {HRF_SPMG1.nbasis}")
print(f"Span (duration): {HRF_SPMG1.span} seconds")

# Evaluate the HRF at specific time points
time_points = np.arange(0, 20, 0.1)
hrf_values = evaluate(HRF_SPMG1, time_points)

# Plot the HRF
plt.figure(figsize=(10, 6))
plt.plot(time_points, hrf_values, 'b-', linewidth=2)
plt.xlabel('Time (seconds)')
plt.ylabel('HRF Amplitude')
plt.title('SPM Canonical HRF')
plt.grid(True, alpha=0.3)
plt.show()

## Creating Custom HRF Objects

You can create custom HRF objects using various built-in types. The `gen_hrf()` function provides a flexible interface for this:

In [ ]:
# Create a Gaussian HRF
gaussian_hrf = gen_hrf(type="gaussian", width=4, center=5)

# Create a double-gamma HRF with custom parameters
dgamma_hrf = gen_hrf(
    type="dgamma",
    a1=6, a2=16,  # Shape parameters
    b1=1, b2=1,   # Scale parameters
    c=1/6         # Ratio of response to undershoot
)

# Compare different HRFs
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot Gaussian HRF
time_points = np.arange(0, 30, 0.1)
gaussian_values = evaluate(gaussian_hrf, time_points)
ax1.plot(time_points, gaussian_values, 'g-', linewidth=2)
ax1.set_xlabel('Time (seconds)')
ax1.set_ylabel('HRF Amplitude')
ax1.set_title('Gaussian HRF')
ax1.grid(True, alpha=0.3)

# Plot Double-Gamma HRF
dgamma_values = evaluate(dgamma_hrf, time_points)
ax2.plot(time_points, dgamma_values, 'r-', linewidth=2)
ax2.set_xlabel('Time (seconds)')
ax2.set_ylabel('HRF Amplitude')
ax2.set_title('Double-Gamma HRF')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## HRF with Temporal Derivatives

For more flexible modeling, you can include temporal and dispersion derivatives. This creates a basis set that can capture variations in the timing and shape of the hemodynamic response:

In [ ]:
# Create HRF with temporal derivative
hrf_with_deriv = gen_hrf(
    type="spmg1",
    include_deriv=1  # Include first temporal derivative
)

print(f"Number of basis functions: {hrf_with_deriv.nbasis}")

# Evaluate all basis functions
time_points = np.arange(0, 20, 0.1)
basis_functions = evaluate(hrf_with_deriv, time_points)

# Plot the basis functions
plt.figure(figsize=(10, 6))
plt.plot(time_points, basis_functions[:, 0], 'b-', linewidth=2, label='Canonical HRF')
plt.plot(time_points, basis_functions[:, 1], 'r--', linewidth=2, label='Temporal Derivative')
plt.xlabel('Time (seconds)')
plt.ylabel('Amplitude')
plt.title('HRF with Temporal Derivative')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Finite Impulse Response (FIR) Models

FIR models make no assumptions about the shape of the HRF, instead using a set of basis functions to estimate the response at each time point:

In [ ]:
# Create FIR basis set
fir_hrf = gen_hrf(
    type="fir",
    nbasis=10,  # Number of time bins
    span=20     # Total duration in seconds
)

# Evaluate FIR basis
time_points = np.arange(0, 20, 0.1)
fir_basis = evaluate(fir_hrf, time_points)

# Plot FIR basis functions
plt.figure(figsize=(12, 8))
for i in range(fir_basis.shape[1]):
    plt.subplot(2, 5, i+1)
    plt.plot(time_points, fir_basis[:, i], linewidth=2)
    plt.title(f'Basis {i+1}')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.grid(True, alpha=0.3)

plt.suptitle('FIR Basis Functions', fontsize=14)
plt.tight_layout()
plt.show()

## Convolution: From Neural Events to BOLD Signal

The key operation in fMRI analysis is convolving neural events with the HRF to predict the BOLD signal. Let's demonstrate this with a simple example:

In [ ]:
# Create a sampling frame (100 scans, TR=2 seconds)
sframe = SamplingFrame(n_scans=100, tr=2.0)

# Define event onsets (in seconds)
event_onsets = [10, 30, 50, 70, 90]

# Create regressor by convolving events with HRF
reg = regressor(
    onsets=event_onsets,
    hrf=HRF_SPMG1,
    sampling_frame=sframe
)

# Get the time points for plotting
time_samples = samples(sframe)

# Plot the result
plt.figure(figsize=(12, 6))

# Plot events as vertical lines
for onset in event_onsets:
    plt.axvline(x=onset, color='red', linestyle='--', alpha=0.5, label='Event' if onset == event_onsets[0] else '')

# Plot convolved signal
plt.plot(time_samples, reg, 'b-', linewidth=2, label='Convolved BOLD Signal')

plt.xlabel('Time (seconds)')
plt.ylabel('Signal Amplitude')
plt.title('Neural Events Convolved with HRF')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Comparing Different HRFs

Let's compare how different HRF models affect the predicted BOLD signal for the same set of events:

In [ ]:
# Define HRFs to compare
hrfs = {
    'SPM Canonical': HRF_SPMG1,
    'Gaussian': gen_hrf(type="gaussian", width=4, center=5),
    'Gamma': gen_hrf(type="gamma", shape=6, scale=1),
    'Double Gamma': gen_hrf(type="dgamma", a1=6, a2=16, b1=1, b2=1, c=1/6)
}

# Create sampling frame
sframe = SamplingFrame(n_scans=150, tr=2.0)
time_samples = samples(sframe)

# More complex event pattern
event_onsets = [10, 15, 40, 45, 70, 100, 105, 110, 140]

# Plot comparison
plt.figure(figsize=(14, 10))

for i, (name, hrf) in enumerate(hrfs.items(), 1):
    plt.subplot(2, 2, i)
    
    # Create regressor
    reg = regressor(onsets=event_onsets, hrf=hrf, sampling_frame=sframe)
    
    # Plot events
    for onset in event_onsets:
        plt.axvline(x=onset, color='red', linestyle='--', alpha=0.3)
    
    # Plot convolved signal
    plt.plot(time_samples, reg, 'b-', linewidth=2)
    plt.title(f'{name} HRF')
    plt.xlabel('Time (seconds)')
    plt.ylabel('Signal Amplitude')
    plt.grid(True, alpha=0.3)

plt.suptitle('Comparison of Different HRF Models', fontsize=16)
plt.tight_layout()
plt.show()

## Summary

In this notebook, we've covered:

1. **Basic HRF concepts** - Understanding how HRFs model the BOLD response
2. **Pre-defined HRFs** - Using built-in HRF models like SPM canonical
3. **Custom HRFs** - Creating HRFs with specific parameters
4. **HRF derivatives** - Adding temporal flexibility with derivative terms
5. **FIR models** - Non-parametric approach to HRF estimation
6. **Convolution** - Converting neural events to predicted BOLD signals
7. **HRF comparison** - Understanding how HRF choice affects the model

The choice of HRF can significantly impact your fMRI analysis. The SPM canonical HRF with derivatives is a good default choice for most studies, but FIR models can be useful when the HRF shape is unknown or expected to vary.